# 面试问题：Reward Model、Bradley–Terry Loss 和 KL 约束是什么？

可直接复述的回答：Reward Model 把 prompt-response 映射成标量，但标量只在同一训练定义下有意义。偏好对用 Bradley–Terry 概率表示 chosen reward 高于 rejected reward，稳定损失是 softplus 负 margin。数据要按用户或 prompt group 切分，避免同题改写泄漏。评估不仅看 pair accuracy，还看 margin、校准和安全切片。策略优化不能无限追逐 reward，应对参考策略施加 KL 约束。KL 系数控制收益与分布漂移的权衡。Reward hacking 需要独立安全特征、红队集和上线门禁。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：配送客服偏好对与输入预览

六组脱敏偏好对用三个可解释特征表示：有用性、简洁性和风险。chosen/rejected 来自同一用户问题，保留了为什么人工偏好的语义；特征只是教学代理，不是实际 embedding。


In [1]:
import torch  # 使用 PyTorch 基础张量实现奖励训练。
import torch.nn.functional as F  # 使用 softplus 和 softmax 构造稳定目标。
torch.manual_seed(20260729)  # 固定训练初始化和输出。
pairs11 = [  # 构造六组客服回答偏好特征。
    {"id": "late-1", "chosen": [0.95, 0.80, 0.00], "rejected": [0.45, 0.40, 0.00], "reason": "给出查询步骤"},  # 有用回答胜过空泛安慰。
    {"id": "refund-1", "chosen": [0.90, 0.75, 0.00], "rejected": [0.92, 0.20, 0.70], "reason": "拒绝泄露内部凭据"},  # 略少细节但安全。
    {"id": "address-1", "chosen": [0.82, 0.90, 0.00], "rejected": [0.80, 0.25, 0.00], "reason": "避免重复解释"},  # 简洁回答更受偏好。
    {"id": "coupon-1", "chosen": [0.88, 0.70, 0.00], "rejected": [0.55, 0.85, 0.00], "reason": "规则正确优先"},  # 正确性胜过短但错误。
    {"id": "damage-1", "chosen": [0.93, 0.65, 0.00], "rejected": [0.96, 0.30, 0.80], "reason": "不要求上传身份证"},  # 风险特征用于防 reward hacking。
    {"id": "eta-1", "chosen": [0.86, 0.85, 0.00], "rejected": [0.60, 0.50, 0.00], "reason": "说明时间范围"},  # 可执行预期优于模糊回答。
]  # 完成六条有业务理由的偏好记录。
print("教学实验输入：id | chosen特征 | rejected特征 | 人工理由")  # 输出偏好数据表头。
for pair11 in pairs11:  # 逐条展示偏好对和理由。
    print(pair11)  # 输出一组人工偏好。


教学实验输入：id | chosen特征 | rejected特征 | 人工理由
{'id': 'late-1', 'chosen': [0.95, 0.8, 0.0], 'rejected': [0.45, 0.4, 0.0], 'reason': '给出查询步骤'}
{'id': 'refund-1', 'chosen': [0.9, 0.75, 0.0], 'rejected': [0.92, 0.2, 0.7], 'reason': '拒绝泄露内部凭据'}
{'id': 'address-1', 'chosen': [0.82, 0.9, 0.0], 'rejected': [0.8, 0.25, 0.0], 'reason': '避免重复解释'}
{'id': 'coupon-1', 'chosen': [0.88, 0.7, 0.0], 'rejected': [0.55, 0.85, 0.0], 'reason': '规则正确优先'}
{'id': 'damage-1', 'chosen': [0.93, 0.65, 0.0], 'rejected': [0.96, 0.3, 0.8], 'reason': '不要求上传身份证'}
{'id': 'eta-1', 'chosen': [0.86, 0.85, 0.0], 'rejected': [0.6, 0.5, 0.0], 'reason': '说明时间范围'}


## 2. Baseline（基线）：只奖励有用性

只看第一个特征会偏爱包含更多操作细节的危险回答，在 `refund-1` 和 `damage-1` 上发生 reward hacking。


In [2]:
baseline_weight11 = torch.tensor([1.0, 0.0, 0.0])  # 定义只看有用性的朴素奖励权重。
baseline_rows11 = []  # 收集基线偏好判断。
for pair11 in pairs11:  # 对每组回答计算朴素 reward margin。
    chosen11 = torch.tensor(pair11["chosen"], dtype=torch.float32)  # 转换 chosen 特征。
    rejected11 = torch.tensor(pair11["rejected"], dtype=torch.float32)  # 转换 rejected 特征。
    margin11 = float((chosen11 - rejected11) @ baseline_weight11)  # 计算只看有用性的奖励差。
    baseline_rows11.append((pair11["id"], round(margin11, 3), margin11 > 0.0))  # 保存 margin 和偏好是否命中。
baseline_accuracy11 = sum(row11[2] for row11 in baseline_rows11) / len(baseline_rows11)  # 计算基线 pair accuracy。
print("基线：id | chosen-rejected margin | 命中人工偏好")  # 输出基线结果表头。
for row11 in baseline_rows11:  # 逐条展示 reward hacking 样本。
    print(row11)  # 输出一组基线 margin。


基线：id | chosen-rejected margin | 命中人工偏好
('late-1', 0.5, True)
('refund-1', -0.02, False)
('address-1', 0.02, True)
('coupon-1', 0.33, True)
('damage-1', -0.03, False)
('eta-1', 0.26, True)


## 3. 核心实现：手写 Bradley–Terry Loss

线性 Reward Model 直接学习三个特征权重。对每对回答计算 margin，再最小化 `softplus(-margin)`；这是 `-log sigmoid(margin)` 的数值稳定形式。


In [3]:
chosen_matrix11 = torch.tensor([pair11["chosen"] for pair11 in pairs11], dtype=torch.float32)  # 组成 chosen 特征矩阵。
rejected_matrix11 = torch.tensor([pair11["rejected"] for pair11 in pairs11], dtype=torch.float32)  # 组成 rejected 特征矩阵。
weight11 = torch.zeros(3, requires_grad=True)  # 初始化可训练线性奖励权重。
optimizer11 = torch.optim.Adam([weight11], lr=0.12)  # 使用基础优化器更新权重。
loss_trace11 = []  # 保存 Bradley–Terry loss 曲线。
for _ in range(120):  # 运行确定性小型偏好训练。
    optimizer11.zero_grad()  # 清空上一步奖励梯度。
    chosen_reward11 = chosen_matrix11 @ weight11  # 计算 chosen 标量奖励。
    rejected_reward11 = rejected_matrix11 @ weight11  # 计算 rejected 标量奖励。
    margins11 = chosen_reward11 - rejected_reward11  # 计算每组偏好 margin。
    loss11 = F.softplus(-margins11).mean() + 0.005 * weight11.square().sum()  # 计算稳定 BT loss 并轻微正则化。
    loss11.backward()  # 反向传播奖励模型梯度。
    optimizer11.step()  # 更新三个可解释权重。
    loss_trace11.append(float(loss11.detach()))  # 保存当前损失。
trained_margins11 = (chosen_matrix11 @ weight11.detach()) - (rejected_matrix11 @ weight11.detach())  # 计算训练后的成对 margin。
pair_probabilities11 = torch.sigmoid(trained_margins11)  # 转换为 chosen 获胜概率。
print("BT训练：loss start -> end", round(loss_trace11[0], 4), "->", round(loss_trace11[-1], 4))  # 展示训练过程变化。
print("学习到的权重 [有用,简洁,风险]", [round(float(value11), 3) for value11 in weight11.detach()])  # 展示风险权重方向。
print("逐对 chosen 概率", [(pairs11[index11]["id"], round(float(probability11), 3)) for index11, probability11 in enumerate(pair_probabilities11)])  # 展示每组偏好置信度。


BT训练：loss start -> end 0.6931 -> 0.2627
学习到的权重 [有用,简洁,风险] [3.091, 2.829, -1.84]
逐对 chosen 概率 [('late-1', 0.936), ('refund-1', 0.942), ('address-1', 0.87), ('coupon-1', 0.645), ('damage-1', 0.914), ('eta-1', 0.857)]


## 4. 结果表、KL 策略与结果解读

训练后风险权重应为负，所有 chosen margin 为正。随后对三个候选回答做解析 KL 更新：`pi ∝ pi_ref * exp(reward/beta)`；beta 越小，策略越激进地追逐 reward。


In [4]:
trained_accuracy11 = float((trained_margins11 > 0.0).float().mean())  # 计算 Reward Model pair accuracy。
candidate_names11 = ["safe_short", "safe_detailed", "unsafe_verbose"]  # 定义同一 prompt 的三个策略候选。
candidate_features11 = torch.tensor([[0.76, 0.95, 0.00], [0.94, 0.65, 0.00], [0.98, 0.20, 0.85]], dtype=torch.float32)  # 描述候选的有用、简洁与风险。
candidate_rewards11 = candidate_features11 @ weight11.detach()  # 使用训练后的 Reward Model 打分。
reference_probs11 = torch.tensor([0.40, 0.45, 0.15], dtype=torch.float32)  # 定义参考策略概率。
beta11 = 0.8  # 设置教学用 KL 约束强度。
policy_probs11 = torch.softmax(torch.log(reference_probs11) + candidate_rewards11 / beta11, dim=0)  # 计算 KL 正则后的最优离散策略。
aggressive_probs11 = torch.softmax(torch.log(reference_probs11) + candidate_rewards11 / 0.15, dim=0)  # 计算较弱 KL 下的激进策略。
print("方法 | pair accuracy | loss_end")  # 输出 Reward Model 对照表头。
print("helpfulness_only", round(baseline_accuracy11, 3), "未训练")  # 展示单特征基线。
print("bradley_terry", round(trained_accuracy11, 3), round(loss_trace11[-1], 4))  # 展示训练后成对结果。
print("KL策略：候选 | reward | reference | beta=0.8 | beta=0.15")  # 输出策略权衡表头。
for index11, name11 in enumerate(candidate_names11):  # 逐候选展示奖励与概率漂移。
    print(name11, round(float(candidate_rewards11[index11]), 3), round(float(reference_probs11[index11]), 3), round(float(policy_probs11[index11]), 3), round(float(aggressive_probs11[index11]), 3))  # 输出一个候选的完整路由概率。
print("结果解读：负风险权重压制危险回答，KL让策略保留参考分布支持")  # 解释奖励与 KL 的不同职责。


方法 | pair accuracy | loss_end
helpfulness_only 0.667 未训练
bradley_terry 1.0 0.2627
KL策略：候选 | reward | reference | beta=0.8 | beta=0.15
safe_short 5.036 0.4 0.559 0.862
safe_detailed 4.744 0.45 0.436 0.138
unsafe_verbose 2.03 0.15 0.005 0.0
结果解读：负风险权重压制危险回答，KL让策略保留参考分布支持


## 5. 失败案例与修正：冗长危险回答奖励更高

基线只看有用性时，危险回答的 0.98 高于安全详细回答的 0.94。加入人工安全偏好后，负风险权重应让危险回答不再是最高 reward；KL 仍不能替代安全门禁。


In [5]:
baseline_candidate_rewards11 = candidate_features11 @ baseline_weight11  # 用单一有用性奖励候选。
baseline_choice11 = candidate_names11[int(torch.argmax(baseline_candidate_rewards11))]  # 获取 reward hacking 下的选择。
fixed_choice11 = candidate_names11[int(torch.argmax(candidate_rewards11))]  # 获取多特征 Reward Model 的选择。
print("失败行为：单特征reward选择", baseline_choice11, [round(float(value11), 3) for value11 in baseline_candidate_rewards11])  # 展示危险回答获胜。
print("修正行为：BT reward选择", fixed_choice11, [round(float(value11), 3) for value11 in candidate_rewards11])  # 展示风险偏好训练后的选择。
print("额外门禁：任何 risk > 0.5 的候选在策略采样前直接移除")  # 强调安全约束不应只依赖标量 reward。


失败行为：单特征reward选择 unsafe_verbose [0.76, 0.94, 0.98]
修正行为：BT reward选择 safe_short [5.036, 4.744, 2.03]
额外门禁：任何 risk > 0.5 的候选在策略采样前直接移除


## 6. 生产边界与奖励制品

真实 Reward Model 使用文本编码器并按 prompt group 切分，还要处理标注者差异、tie 和分布漂移。KL 通常按 token 估计，beta 需要自适应监控；安全策略必须独立于 reward。


In [6]:
reward_contract11 = {"preference_set": "support-prefs-v6", "features": ["helpfulness", "conciseness", "risk"], "loss": "bradley_terry", "beta": beta11, "safety_gate": "risk<=0.5"}  # 定义奖励和策略发布合同。
print("Reward/KL 发布制品", reward_contract11)  # 展示数据、loss、KL和安全门禁版本。
print("生产替换点：文本编码器、group split、tie模型、校准、token级KL和独立红队安全门禁")  # 说明线性特征教学模型的边界。


Reward/KL 发布制品 {'preference_set': 'support-prefs-v6', 'features': ['helpfulness', 'conciseness', 'risk'], 'loss': 'bradley_terry', 'beta': 0.8, 'safety_gate': 'risk<=0.5'}
生产替换点：文本编码器、group split、tie模型、校准、token级KL和独立红队安全门禁


## 7. 最小回归测试

断言保护 BT 训练、风险方向、KL 概率和 reward hacking 修正。


In [7]:
assert len(pairs11) >= 5  # 保证偏好数据覆盖多个业务理由。
assert loss_trace11[-1] < loss_trace11[0]  # 保证 Bradley–Terry 训练能够收敛。
assert trained_accuracy11 > baseline_accuracy11  # 保证多特征奖励优于单一有用性基线。
assert float(weight11.detach()[2]) < 0.0  # 保证风险特征获得负奖励方向。
assert abs(float(policy_probs11.sum()) - 1.0) < 1e-6  # 保证 KL 策略概率归一化。
assert baseline_choice11 == "unsafe_verbose" and fixed_choice11 != "unsafe_verbose"  # 保证 reward hacking 反例被修正。
print("最小回归测试通过：BT margin、风险权重、KL概率和安全反例稳定")  # 显示奖励学习关键性质已验证。


最小回归测试通过：BT margin、风险权重、KL概率和安全反例稳定
